## 🧠 内存管理与性能监控


在深度学习中，GPU 显存（VRAM）是最容易碰到的瓶颈。为了让你彻底理解 PyTorch 的显存管理机制，我们需要先搞懂一个核心概念：“已分配显存（Allocated）”与“已预留显存（Reserved）”。

- <font color='red'>Allocated（已分配）</font>：你的张量（Tensor）实实在在占用的显存。
- <font color='red'>Reserved（已预留）</font>：<font color='green'>PyTorch 为了加速，向显卡驱动提前“圈地”占用的显存池。即使你 del 删除了张量，这部分显存 PyTorch 也会暂时扣着不放，方便下次快速复用。</font>

理解了这一点，下面这四个方法的作用就非常清晰了：

## 🧹 torch.cuda.empty_cache()：<font color='red'>释放“闲置”的预留显存</font>
它的作用是强制把 PyTorch 缓存分配器中未占用（即 Reserved 减去 Allocated 的差值）的显存还给系统。
- 作用：调用后，你在 nvidia-smi 里看到的显存占用会明显下降，其他进程可以使用这部分显存。
- 注意：它不能释放被张量实际占用的显存。<font color='red'>频繁调用它会降低程序运行效率（因为下次分配显存需要重新向系统申请），通常只在模型验证阶段或长任务间隙使用。</font>

## 📊 torch.cuda.memory_allocated()：查看当前“实际”用了多少
- 返回当前指定设备上，所有张量实实在在占用的显存字节数。这是判断你的模型和 Batch Size 到底需要多少显存的最准确指标。
## 📈 torch.cuda.max_memory_allocated()：<font color='red'>排查“爆显存”的元凶</font>
- 返回程序从开始运行到现在，曾经占用过的最大显存字节数。如果你遇到了 CUDA Out of Memory (OOM) 报错，这个指标能告诉你程序运行的峰值显存到底到了多少，方便你针对性优化。

## 📋 torch.cuda.memory_summary()：打印完整的显存体检报告
- 以表格形式打印出当前设备的详细显存分配、预留、碎片化程度等摘要信息，适合在调试时全面评估显存状态。


In [ ]:
import torch
import time

# 检查是否有可用 GPU
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    
    print("--- 1. 初始状态 ---")
    print(f"实际占用: {torch.cuda.memory_allocated(device) / 1024**2:.2f} MB")
    print(f"历史峰值: {torch.cuda.max_memory_allocated(device) / 1024**2:.2f} MB\n")

    # 模拟创建一个 500MB 左右的大张量 (10000 * 10000 * 4字节(float32) / 1024^2 ≈ 381MB，这里适当加大)
    print("--- 2. 创建大张量后 ---")
    big_tensor = torch.randn(20000, 20000, device=device) 
    print(f"实际占用: {torch.cuda.memory_allocated(device) / 1024**2:.2f} MB")
    print(f"历史峰值: {torch.cuda.max_memory_allocated(device) / 1024**2:.2f} MB\n")

    # 删除张量
    print("--- 3. 删除张量 (del) 但未清理缓存 ---")
    del big_tensor
    print(f"实际占用: {torch.cuda.memory_allocated(device) / 1024**2:.2f} MB (已归零)")
    print(f"历史峰值: {torch.cuda.max_memory_allocated(device) / 1024**2:.2f} MB (峰值记录依然保留)\n")
    
    # 此时如果去系统里看 nvidia-smi，显存可能还没降下来，因为 PyTorch 预留了缓存
    
    time.sleep(2) # 暂停一下，方便观察
    
    # 强制释放缓存
    print("--- 4. 调用 empty_cache() 强制释放缓存后 ---")
    torch.cuda.empty_cache()
    print(f"实际占用: {torch.cuda.memory_allocated(device) / 1024**2:.2f} MB")
    print(f"历史峰值: {torch.cuda.max_memory_allocated(device) / 1024**2:.2f} MB")
    
    # 打印详细摘要（输出内容较多，通常在需要深度排查时打开）
    # print("\n--- 5. 显存详细摘要 ---")
    # print(torch.cuda.memory_summary(device))

else:
    print("当前没有可用的 GPU。")

## 💡 最佳实践建议：
- <font color='green'>在日常训练脚本中，你可以在每个 Epoch 结束或者进入验证（Validation）阶段前，调用一次 torch.cuda.empty_cache()，这样既能保证训练进程不占用多余显存，也能让系统状态更干净。</font>
- 而 memory_allocated 和 max_memory_allocated 则非常适合写在你的训练日志里，帮你监控显存的健康度。